In [163]:
import numpy as np
from sympy.polys.benchmarks.bench_solvers import time_eqs_10x8
#pip install ultralytics

In [ ]:
import torch
from ultralytics import YOLO
import numpy as np
model = YOLO(r"runs/obb/train-5/weights/best.pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


In [165]:
#pip install mss

In [166]:
#pip install pygetwindow

In [ ]:
#pip install climage
!pip install pyautogui

In [ ]:
import pygetwindow as gw
import mss
import climage
import cv2 as cv
import time
import pyautogui
from pynput import keyboard
from pynput.keyboard import Key, Controller



# this block locates the postion of the vampire survivors window
app_title = "Vampire Survivors"
try:
    window = gw.getWindowsWithTitle(app_title)
except IndexError:
    print("Whoops")
print(window[0])
vp_box = {
    "left": window[0].left,
    "top": window[0].top,
    "width": window[0].width,
    "height": window[0].height,
}
counter = 0

tO = time.time()
n_frames = 1



# these are some average values, they are mostly irrelevant because they get almost
# instantly overwritten
smallest_difference_height = 6

smallest_difference_weight = 6

swdistance_y = 5
distance_x = 5



# this is the main loop that contains the screenshots and the logic
with (mss.MSS() as sct):
    while True:
        controller = keyboard.Controller()
        controller.release ('w')
        controller.release ('a')
        controller.release ('s')
        controller.release ('d')

        img = sct.grab(vp_box)

        if n_frames == 1:
            w, h = img.size
            w = w/2
            h = h/2
            center_of_the_screen_height =  (h)
            center_of_the_screen_weight =  (w)
            player = (int(w), int(h))

        img = np.array(img)

        small = cv.resize(img, (0,0), fx = 1, fy = 1)

        time.sleep(0)
        if small.shape[-1] == 4:
            small = small[:, :, :3]

        results = model(small, verbose=False, save = False, name = "ss", conf=0.70)
        plotted_img = results[0].plot(labels=True, boxes=True, line_width=1)


        elapsed_time = time.time() - tO
        avg_fps = (n_frames / elapsed_time)

        n_frames += 1

        for r in results:
         x_distance_away = -800
         y_distance_away = -800
         new_total_distance = 8000

         current_tensors = r.obb.xyxy



         for x in current_tensors:
             upper_left_corner_x_value = x[0]
             lower_right_corner_x_value = x[2]
             upper_left_corner_y_value = x[1]
             lower_right_corner_y_value = x[3]

             distance_x =  (upper_left_corner_x_value + lower_right_corner_x_value)/2
             distance_y =  (upper_left_corner_y_value + lower_right_corner_y_value)/2

             total_distance =  np.sqrt((center_of_the_screen_weight  - distance_x )**2 + (center_of_the_screen_height  - distance_y )**2)

             if total_distance < new_total_distance:
                 new_total_distance = total_distance
                 x_distance_away = distance_x
                 y_distance_away = distance_y


        mob = (int(x_distance_away), int(y_distance_away))
        cv.line(plotted_img, player, mob, (0, 255, 0), thickness=3)

        if y_distance_away > center_of_the_screen_height:
           controller.press ('w')
        else:
           controller.press ('s')
        if x_distance_away > center_of_the_screen_weight:
           controller.press ('a')
        else:
           controller.press ('d')
        time.sleep(0.1)
        cv.imshow("Computer Vision", plotted_img)
        cv.waitKey(1)


cv.destroyAllWindows()